In [1]:
import synapseclient
syn = synapseclient.Synapse()
import synapseutils
import pandas as pd
import os
import numpy as np
import pandas as pd


In [ ]:
input_dir = './input'
summary_stats_dir = './summary_stats'
pqtl_dir = os.path.join(summary_stats_dir, 'pqtls_as_exposures')

os.makedirs(pqtl_dir, exist_ok=True)
os.getcwd()

# Downloading from Synapse

In [ ]:
syn.login(authToken="INSERT_TOKEN_HERE")

synapse_filesnames = pd.read_csv(os.path.join(input_dir, 'all_synapse_ukb_ppp_filenames.csv'))

#filtering to only the European discovery gwas files
synapse_filesnames = synapse_filesnames[synapse_filesnames['parentId'] == 'syn51365303']

protein_codes = pd.read_csv(os.path.join(input_dir, 'olink_protein_map_3k_v1.tsv'), sep='\t')


selected_proteins = pd.read_csv('./selection_output/selected_proteins.csv')


In [4]:
selected_proteins['feature'] = selected_proteins['feature'].apply(lambda x: ''.join([char.upper() if char.isalpha() else char for char in x]))

synapse_filesnames['prefix'] = synapse_filesnames['name'].apply(lambda x: x.split('_')[0])


In [ ]:
unique_features = selected_proteins['feature'].unique()
print(len(unique_features))

directory_path = pqtl_dir

files_in_directory = os.listdir(directory_path)

# Extract the unique features present in the directory
existing_features = set()
for filename in files_in_directory:
    feature_code = filename.split('_')[0]
    existing_features.add(feature_code)

# Determine which features are not yet present
unique_features_remainder = [feature for feature in unique_features if feature not in existing_features]

unique_features_remainder = np.array(unique_features_remainder)

print(unique_features_remainder)
print(len(unique_features_remainder))

synapse_filesnames = synapse_filesnames[synapse_filesnames['prefix'].isin(unique_features_remainder)]
print(len(synapse_filesnames))

In [ ]:
# Retrieve unique IDs
ids = synapse_filesnames['id'].unique()
path = pqtl_dir

for entity_id in ids:
    print(f'Syncing files for {entity_id}...')
    all_files = synapseutils.syncFromSynapse(syn, entity=entity_id, path=path)
    print(f'Finished syncing files for {entity_id}')